# Phase 2: Resume & Job Description Preprocessing

This notebook demonstrates the preprocessing pipeline for resumes and job descriptions.

In [1]:
import os
import sys
import pandas as pd

# Add project root to path (Colab-friendly)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from datasets import load_dataset
from src.preprocessing.text_cleaner import (
    clean_text, 
    preprocess_resume_record, 
    preprocess_job_record, 
    detect_and_flag_duplicates
)

pd.set_option('display.max_colwidth', 100)


## 2. Dataset Loading

Loading a subset of datasets to demonstrate the preprocessing pipeline scalable to larger datasets.

In [2]:
# 1. Resume-Job Fit Merged v1 (Primary dataset)
ds_fit = load_dataset("med2425/resume-job-fit-merged-v1", split="train")
df_fit_sample = ds_fit.select(range(100)).to_pandas()

# 2. EXAI-ResumeIntel Resumes
ds_resumes = load_dataset("mithinsagar/exai-resumeintel-data", "resumes", split="train")
df_resumes_sample = ds_resumes.select(range(100)).to_pandas()

# 3. EXAI-ResumeIntel Job Postings (Streaming sample)
ds_jobs_stream = load_dataset("mithinsagar/exai-resumeintel-data", "job_postings", split="train", streaming=True)
jobs_sample = []
for i, record in enumerate(ds_jobs_stream):
    if i >= 100: break
    jobs_sample.append(record)
df_jobs_sample = pd.DataFrame(jobs_sample)

print(f"Fit Sample: {df_fit_sample.shape}")
print(f"Resumes Sample: {df_resumes_sample.shape}")
print(f"Jobs Sample: {df_jobs_sample.shape}")


Fit Sample: (100, 6)
Resumes Sample: (100, 3)
Jobs Sample: (100, 2)


## 3. Raw Data Inspection

In [3]:
display(df_fit_sample.head(2))
display(df_resumes_sample.head(2))
display(df_jobs_sample.head(2))


,resume,jd,label,source,resume_domain,jd_domain
0,SummarySelf-motivated accountant offering a strong work ethic and determination to complete tas...,"Our client, a Raleigh-based full-service commercial real estate partner, is actively recruiting ...",Potential Fit,generated_smart,finance,finance
1,"Professional ProfileSpecialized knowledge of regulations, policies, priorities, commitments and ...","""All candidates must be directly contracted by ASK Consulting on their payroll and cannot be sub...",No Fit,generated_smart,finance,software


,ID,Category,Feature
0,16852973,HR,hr administrator marketing associate hr administrator summary dedicated customer service manager...
1,22323967,HR,hr specialist hr operations summary media professional background communications marketing human...


,Role,Features
0,Social Media Manager,"5 to 15 Years Digital Marketing Specialist M.Tech Social media platforms (e.g., Facebook, Twitte..."
1,Frontend Web Developer,"2 to 12 Years Web Developer BCA HTML, CSS, JavaScript Frontend frameworks (e.g., React, Angular)..."


## 4. Text-Field Identification

- Fit Dataset: `resume`, `jd`
- Resumes Dataset: `Feature`
- Jobs Dataset: `Features`
We map these to standard keys for our pipeline.

## 5. Missing-Value Handling

Our `handle_nulls` and `clean_text` functions automatically handle NaNs and None types by converting them to empty strings.

In [4]:
# Simulate a missing value to demonstrate
test_df = pd.DataFrame({"text": ["Valid text", None, float('nan'), ""]})
test_df["cleaned"] = test_df["text"].apply(clean_text)
display(test_df)


,text,cleaned
0,Valid text,Valid text
1,NaN,
2,NaN,
3,,


## 6. Duplicate Handling

We identify exact duplicates without silently deleting them, allowing downstream processes to use the `duplicate_group` column for train/test splits to prevent leakage.

In [5]:
# Inject a duplicate for demonstration
df_fit_sample.loc[1, 'resume'] = df_fit_sample.loc[0, 'resume']

# Flag duplicates
df_fit_sample = detect_and_flag_duplicates(df_fit_sample, text_column='resume')
print("Duplicate Groups Found:", df_fit_sample[df_fit_sample['duplicate_group'] != ""]['duplicate_group'].unique())


Duplicate Groups Found: <ArrowStringArray>
['0142c9a4', '52e2c364', 'eeec880f', '1074bc9d', '05205909', '46120292',
 '98515d68', '8ebbcc79', 'fe80963b', 'b248b8e3', 'a3247867', 'd9fc39a7']
Length: 12, dtype: str


## 7 & 8. Text Cleaning and Normalization

Whitespace, control characters, and duplicated punctuations are normalized. We explicitly preserve technical terms.

In [6]:
tech_text = "Experienced in C++, C#, .NET, React.js, CI/CD \n\n *** Need AWS & GCP!!!"
print("Raw:", repr(tech_text))
print("Cleaned:", repr(clean_text(tech_text)))


Raw: 'Experienced in C++, C#, .NET, React.js, CI/CD \n\n *** Need AWS & GCP!!!'
Cleaned: 'Experienced in C++, C#, .NET, React.js, CI/CD Need AWS & GCP!'


## 9. Resume Preprocessing

Applying the `preprocess_resume_record` function.

In [7]:
# Using the fit dataset resumes
processed_resumes = [preprocess_resume_record(row.to_dict(), text_column="resume") for _, row in df_fit_sample.iterrows()]
df_processed_resumes = pd.DataFrame(processed_resumes)
display(df_processed_resumes[['raw_text', 'cleaned_resume_text', 'resume_word_count']].head(2))


,raw_text,cleaned_resume_text,resume_word_count
0,SummarySelf-motivated accountant offering a strong work ethic and determination to complete tas...,SummarySelf-motivated accountant offering a strong work ethic and determination to complete task...,689
1,SummarySelf-motivated accountant offering a strong work ethic and determination to complete tas...,SummarySelf-motivated accountant offering a strong work ethic and determination to complete task...,689


## 10. Job-Description Preprocessing

Applying the `preprocess_job_record` function.

In [8]:
processed_jobs = [preprocess_job_record(row.to_dict(), text_column="jd") for _, row in df_fit_sample.iterrows()]
df_processed_jobs = pd.DataFrame(processed_jobs)
display(df_processed_jobs[['raw_text', 'cleaned_job_text', 'job_word_count']].head(2))


,raw_text,cleaned_job_text,job_word_count
0,"Our client, a Raleigh-based full-service commercial real estate partner, is actively recruiting ...","Our client, a Raleigh-based full-service commercial real estate partner, is actively recruiting ...",279
1,"""All candidates must be directly contracted by ASK Consulting on their payroll and cannot be sub...","""All candidates must be directly contracted by ASK Consulting on their payroll and cannot be sub...",316


## 11. Before/After Examples

In [9]:
sample_record = df_processed_resumes.iloc[0]
print("--- BEFORE ---")
print(sample_record['raw_text'][:300] + "...")
print("\n--- AFTER ---")
print(sample_record['cleaned_resume_text'][:300] + "...")


--- BEFORE ---
SummarySelf-motivated accountant  offering a strong work ethic and determination to complete tasks in a timely manner. Accurate and detail-oriented with extensive auditing and finance knowledge.
HighlightsComplex problem solvingStrong communication skillsExpert in customer relationsPortfolio managem...

--- AFTER ---
SummarySelf-motivated accountant offering a strong work ethic and determination to complete tasks in a timely manner. Accurate and detail-oriented with extensive auditing and finance knowledge. HighlightsComplex problem solvingStrong communication skillsExpert in customer relationsPortfolio manageme...


## 12. Text Statistics

In [10]:
print("Resume Word Count Summary:")
print(df_processed_resumes['resume_word_count'].describe())
print("\nJob Word Count Summary:")
print(df_processed_jobs['job_word_count'].describe())


Resume Word Count Summary:
count     100.000000
mean      775.520000
std       417.733337
min       182.000000
25%       553.750000
50%       665.500000
75%       849.000000
max      2230.000000
Name: resume_word_count, dtype: float64

Job Word Count Summary:
count     100.000000
mean      362.440000
std       225.422805
min        11.000000
25%       196.000000
50%       301.500000
75%       522.250000
max      1015.000000
Name: job_word_count, dtype: float64


## 13. Data-Quality Validation

Ensure no leakage occurred and no fields were improperly destroyed.

In [11]:
# Verify no Nulls in cleaned text
assert df_processed_resumes['cleaned_resume_text'].isnull().sum() == 0

# Check empty strings
empty_count = (df_processed_resumes['cleaned_resume_text'] == "").sum()
print(f"Empty cleaned resumes: {empty_count}")


Empty cleaned resumes: 0


## 14. Processed Dataset Schema

In [12]:
print("Processed Resume Schema:")
print(df_processed_resumes.dtypes)
print("\nProcessed Job Schema:")
print(df_processed_jobs.dtypes)


Processed Resume Schema:
raw_text                 str
cleaned_resume_text      str
resume_text_length     int64
resume_word_count      int64
resume_domain            str
dtype: object

Processed Job Schema:
raw_text              str
cleaned_job_text      str
job_text_length     int64
job_word_count      int64
jd_domain             str
dtype: object


## 15. Saving/Export Strategy

The processed data can be saved to Parquet for efficient storage, though we skip saving to avoid committing large binaries to git. The schema is unified and ready for Phase 3.

## 16. Phase 2 Conclusions

- A robust text cleaning pipeline was created that handles anomalies while preserving critical technical terms (e.g. C++).
- Standardized schema separates resumes and JDs into `cleaned_resume_text` and `cleaned_job_text`.
- Exact duplicates are flagged using a hash-based grouping ID to avoid data leakage during split generation.
- Code is packaged in `src.preprocessing.text_cleaner` for seamless integration in Phase 3.